In [1]:
import tlc
from pathlib import Path
import cv2
import numpy as np


In [2]:
DATASET_ROOT = Path("/Users/gudbrand/Projects/ChessVision-3LC/data/board_extraction")
DATASET_NAME = "chessvision-segmentation"
TABLE_NAME = "initial"
PROJECT_NAME = "chessvision-segmentation"

In [23]:
image_paths = sorted(map(str, DATASET_ROOT.glob("**/images/*.JPG")))
segmentation_map_paths = sorted(DATASET_ROOT.glob("**/masks/*.png"))

In [24]:
# Build the column of instance segmentations in the format required by 3LC
mask_dicts = []

for mask_path in segmentation_map_paths:
    map_np = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    h, w = map_np.shape
    masks = np.expand_dims(map_np, -1)
    mask_dicts.append(
        {
            "image_height": h,
            "image_width": w,
            "masks": masks,
            "instance_properties":
                {
                    "label": [0],
                },
        }
    )


In [25]:
table_writer = tlc.TableWriter(
    table_name=TABLE_NAME,
    dataset_name=DATASET_NAME,
    project_name=PROJECT_NAME,
    column_schemas={
        "image": tlc.ImagePath("image"),
        "instances": tlc.InstanceSegmentationMasks(
            "instances",
            instance_properties_structure={
                "label": tlc.CategoricalLabel("label", ["chessboard"]),
            },
        ),
    },
    if_exists="rename",
)

In [26]:
table_writer.add_batch(
    {
        "image": image_paths,
        "instances": mask_dicts,
    }
)

In [27]:
table = table_writer.finalize()

In [ ]:
table.url
